# Assignment 6: RAG-Based Machine Learning Knowledge Assistant

**Goal:** Build a Question-Answering system that reads a Machine Learning book (PDF) and answers questions using only the book's content.

**Pipeline:**
```
PDF Book → Extract Text → Clean → Split into Chunks → Embed → FAISS Index
                                                                     ↓
                              User Question → Embed → Search → Top Chunks → LLM → Answer
```

## Step 0 — Install Required Libraries

In [ ]:
!pip install -q pypdf langchain langchain-community langchain-openai faiss-cpu openai numpy

## Step 1 — Imports & Configuration

In [ ]:
import os
import re
import warnings
import numpy as np

from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain.schema import Document
from langchain.prompts import PromptTemplate

warnings.filterwarnings('ignore')

os.environ["OPENAI_API_KEY"] = "sk-your-api-key-here"   # <-- paste your key here

PDF_PATH = "ml_book.pdf"   # <-- path to your ML book PDF

---
## Part 1 — Data Understanding & Preprocessing

### 1.1 Load the PDF

In [ ]:
reader    = PdfReader(PDF_PATH)
all_pages = [page.extract_text() or "" for page in reader.pages]

### 1.2 Explore Document Structure

In [ ]:
char_lengths = [len(p) for p in all_pages]

print(f"Total pages         : {len(all_pages)}")
print(f"Avg chars per page  : {int(np.mean(char_lengths))}")
print(f"Min / Max chars     : {min(char_lengths)} / {max(char_lengths)}")
print(f"Near-empty pages    : {sum(1 for c in char_lengths if c < 50)}")
print("\n--- Sample text (page 5) ---")
print(all_pages[4][:400])

### 1.3 Text Quality Check

In [ ]:
broken_words = sum(len(re.findall(r'\w+-\n\w+', p)) for p in all_pages)
extra_spaces = sum(len(re.findall(r' {3,}', p))     for p in all_pages)
number_only  = sum(1 for p in all_pages if re.fullmatch(r'\s*\d+\s*', p))

print(f"Hyphenated line-breaks : {broken_words}")
print(f"Excessive spaces       : {extra_spaces}")
print(f"Page-number-only pages : {number_only}")

### 1.4 Clean the Text

In [ ]:
def clean_page(text):
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)                    # fix hyphenated words
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)   # remove lone page numbers
    text = re.sub(r' {2,}', ' ', text)                                  # collapse extra spaces
    text = re.sub(r'\n{3,}', '\n\n', text)                             # collapse blank lines
    return text.strip()

cleaned_pages = [clean_page(p) for p in all_pages]

### 1.5 Split Text into Chunks

- **chunk_size = 800** — enough text for meaningful context per chunk  
- **chunk_overlap = 120** — avoids losing context at chunk boundaries

In [ ]:
full_text = "\n\n".join(p for p in cleaned_pages if len(p) > 50)

splitter  = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " "]
)
documents = [Document(page_content=chunk) for chunk in splitter.split_text(full_text)]

print(f"Total chunks : {len(documents)}")
print(f"Avg chars    : {int(np.mean([len(d.page_content) for d in documents]))}")
print("\n--- Sample chunk ---")
print(documents[10].page_content)

---
## Part 2 — Embeddings & Vector Store

**Embedding model:** OpenAI `text-embedding-ada-002` — converts each chunk into a 1536-dimensional vector  
**Vector store:** FAISS — stores all vectors and retrieves the closest matches for any query

In [ ]:
embedding_model = OpenAIEmbeddings()
vector_store    = FAISS.from_documents(documents, embedding_model)

### Sample Retrieval

In [ ]:
test_query  = "How does a neural network learn from data?"
top_results = vector_store.similarity_search_with_score(test_query, k=3)

print(f"Query: \"{test_query}\"\n")
for rank, (doc, score) in enumerate(top_results, 1):
    print(f"Rank {rank} | Score: {score:.4f}")
    print(doc.page_content[:250])
    print("-" * 55)

---
## Part 3 — Retrieval Pipeline

In [ ]:
def fetch_chunks(question, k=5):
    return vector_store.similarity_search_with_score(question, k=k)

### Experiment with k = 3, 5, 7

In [ ]:
exp_query = "What is regularisation and why is it needed?"

for k in [3, 5, 7]:
    hits = fetch_chunks(exp_query, k=k)
    print(f"\n=== k={k} ===")
    for i, (doc, score) in enumerate(hits, 1):
        print(f"  [{i}] Score: {score:.4f} | {doc.page_content[:150]}...")

---
## Part 4 — Answer Generation

### Prompt Design

The prompt instructs the LLM to:
- Answer **only** from the provided reference material  
- Admit when the material doesn't cover the topic (hallucination guard)  
- Keep answers structured and clear

In [ ]:
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""\
You are a helpful tutor specialising in Machine Learning and Data Science.
Answer the student's question using ONLY the reference material provided below.

Rules:
- Do not use any knowledge outside the reference material.
- If the material does not contain the answer, say: "The provided material does not cover this topic."
- Use bullet points or numbered steps when explaining a process.

Reference Material:
{context}

Student Question: {question}

Answer:"""
)

llm = OpenAI(temperature=0.1)

In [ ]:
def answer_question(question, k=5):
    hits     = fetch_chunks(question, k=k)
    context  = "\n\n".join(doc.page_content for doc, _ in hits)
    prompt   = RAG_PROMPT.format(context=context, question=question)
    response = llm.invoke(prompt)
    return response.content if hasattr(response, 'content') else response

### Sample Queries & Answers

In [ ]:
questions = [
    "What is the purpose of an activation function in a neural network?",
    "How does the random forest algorithm reduce overfitting?",
    "What is the difference between supervised and unsupervised learning?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {answer_question(q)}")
    print("-" * 60)

### Hallucination Guard Test

In [ ]:
out_of_scope = "What is the GDP of Germany in 2023?"
print(f"Q: {out_of_scope}")
print(f"A: {answer_question(out_of_scope)}")

---
## Part 5 — End-to-End RAG Application

In [ ]:
def rag_qa(question, k=5, show_sources=False):
    hits     = fetch_chunks(question, k=k)
    context  = "\n\n".join(doc.page_content for doc, _ in hits)
    prompt   = RAG_PROMPT.format(context=context, question=question)
    response = llm.invoke(prompt)
    answer   = response.content if hasattr(response, 'content') else response

    print(f"\nQ: {question}")
    if show_sources:
        print("\n[Sources]")
        for i, (doc, score) in enumerate(hits, 1):
            print(f"  ({i}) Score {score:.4f} | {doc.page_content[:120]}...")
    print(f"\nA: {answer}")
    print("=" * 60)

### Demo — Multiple Queries

In [ ]:
demo_questions = [
    "How does gradient descent update model weights during training?",
    "What is the role of the loss function in machine learning?",
    "Explain how principal component analysis reduces the number of features.",
    "What happens when a model has high bias and low variance?",
    "How does the k-means algorithm assign data points to clusters?",
]

for q in demo_questions:
    rag_qa(q, k=5)

In [ ]:
# With source passages visible
rag_qa("What is the vanishing gradient problem and how is it addressed?", k=5, show_sources=True)

### Interactive Q&A Loop

In [ ]:
while True:
    user_q = input("Ask a question (or type 'exit'): ").strip()
    if not user_q:
        continue
    if user_q.lower() in ("exit", "quit"):
        break
    rag_qa(user_q, k=5)